# Phase 8 Error Analysis

Segment-level analysis of the best Phase 7 LightGBM predictions.

The best/worst SKU tables rank store/SKU pairs with meaningful validation demand only. Low or zero-demand SKUs can produce unstable WAPE denominators, so use the intermittency metrics for pure intermittent-demand analysis.

In [ ]:
from __future__ import annotations

import os
from io import BytesIO
from pathlib import Path

import boto3
import pandas as pd

from retail_demand.analysis.error_analysis import (
    best_and_worst_skus,
    bias_flags,
    per_segment_metrics,
)
from retail_demand.analysis.plots import (
    actual_vs_predicted_scatter,
    error_heatmap,
    worst_sku_time_series,
)
from retail_demand.analysis.segments import (
    assign_intermittency_class,
    assign_promo_flag,
    assign_volume_tier,
)
from retail_demand.config import Settings

pd.set_option("display.max_columns", 80)
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "file:///tmp/retail-demand-mlruns")
BEST_RUN_ID = os.getenv("BEST_RUN_ID", "28cb96fe2b5542c797b048dffbb6eae6")
OUTPUT_DIR = Path(os.getenv("ERROR_ANALYSIS_OUTPUT_DIR", "reports/figures"))
TABLE_DIR = Path(os.getenv("ERROR_ANALYSIS_TABLE_DIR", "reports/tables"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print(f"MLflow run: {BEST_RUN_ID}")

In [ ]:
fixture_predictions = os.getenv("ERROR_ANALYSIS_PREDICTIONS_CSV")
if fixture_predictions:
    preds = pd.read_csv(fixture_predictions, parse_dates=["date"])
else:
    import mlflow

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    artifact_path = mlflow.artifacts.download_artifacts(
        run_id=BEST_RUN_ID, artifact_path="forecast_vs_actual.csv"
    )
    preds = pd.read_csv(artifact_path, parse_dates=["date"])

preds["date"] = pd.to_datetime(preds["date"])
preds.head()

In [ ]:
def _read_csv_env(name: str) -> pd.DataFrame | None:
    path = os.getenv(name)
    if not path:
        return None
    return pd.read_csv(path)


def _read_local_silver_table(table: str) -> pd.DataFrame | None:
    root = Path("data/silver") / table
    if not root.exists():
        return None
    return pd.read_parquet(root)


def _read_s3_silver_table(table: str) -> pd.DataFrame | None:
    settings = Settings()
    if not settings.s3_bucket:
        return None
    client = boto3.client("s3", region_name=settings.aws_region)
    prefix = f"{settings.s3_silver_prefix.strip('/')}/{table}/"
    frames = []
    paginator = client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=settings.s3_bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".parquet"):
                body = client.get_object(Bucket=settings.s3_bucket, Key=key)["Body"].read()
                frames.append(pd.read_parquet(BytesIO(body)))
    return pd.concat(frames, ignore_index=True) if frames else None


def read_dimension(table: str, env_name: str) -> pd.DataFrame | None:
    for loader in (_read_csv_env, lambda _name: _read_local_silver_table(table), lambda _name: _read_s3_silver_table(table)):
        frame = loader(env_name)
        if frame is not None:
            return frame
    return None


stores = read_dimension("stores", "ERROR_ANALYSIS_STORES_CSV")
products = read_dimension("products", "ERROR_ANALYSIS_PRODUCTS_CSV")
sales_history = read_dimension("sales", "ERROR_ANALYSIS_SALES_CSV")
prices = read_dimension("prices", "ERROR_ANALYSIS_PRICES_CSV")

for frame in (stores, products, sales_history, prices):
    if frame is not None:
        for date_col in ("date", "week_start", "open_date"):
            if date_col in frame.columns:
                frame[date_col] = pd.to_datetime(frame[date_col])

print({
    "stores": None if stores is None else len(stores),
    "products": None if products is None else len(products),
    "sales_history": None if sales_history is None else len(sales_history),
    "prices": None if prices is None else len(prices),
})

In [ ]:
enriched = preds.copy()
if products is not None and "category" not in enriched.columns:
    enriched = enriched.merge(products[["sku_id", "category", "subcategory"]], on="sku_id", how="left")
if stores is not None and "store_type" not in enriched.columns:
    enriched = enriched.merge(stores[["store_id", "region", "state", "store_type"]], on="store_id", how="left")

if sales_history is None:
    sales_history = enriched.rename(columns={"y_true": "units_sold"})[
        ["store_id", "sku_id", "date", "units_sold"]
    ]

volume_tiers = assign_volume_tier(sales_history)
intermittency = assign_intermittency_class(sales_history)
enriched = enriched.merge(volume_tiers[["store_id", "sku_id", "volume_tier"]], on=["store_id", "sku_id"], how="left")
enriched = enriched.merge(intermittency[["store_id", "sku_id", "intermittency_class"]], on=["store_id", "sku_id"], how="left")
if prices is not None:
    enriched = assign_promo_flag(enriched, prices)
else:
    enriched["is_promo_day"] = False

enriched.head()

In [ ]:
top_categories = (
    enriched.groupby("category", observed=True)["y_true"].sum().sort_values(ascending=False).head(10).index
    if "category" in enriched.columns
    else []
)
category_metrics = per_segment_metrics(enriched[enriched["category"].isin(top_categories)], ["category"])
store_category_metrics = per_segment_metrics(enriched, ["store_id", "category"])
volume_promo_metrics = per_segment_metrics(enriched, ["volume_tier", "is_promo_day"])
intermittency_metrics = per_segment_metrics(enriched, ["intermittency_class"])

category_metrics.to_csv(TABLE_DIR / "segment_metrics_category.csv", index=False)
store_category_metrics.to_csv(TABLE_DIR / "segment_metrics_store_category.csv", index=False)
volume_promo_metrics.to_csv(TABLE_DIR / "segment_metrics_volume_promo.csv", index=False)
intermittency_metrics.to_csv(TABLE_DIR / "segment_metrics_intermittency.csv", index=False)

category_metrics

In [ ]:
bias_tables = {
    "category": bias_flags(category_metrics),
    "store_category": bias_flags(store_category_metrics),
    "volume_promo": bias_flags(volume_promo_metrics),
    "intermittency": bias_flags(intermittency_metrics),
}
for name, table in bias_tables.items():
    table.to_csv(TABLE_DIR / f"bias_flags_{name}.csv", index=False)

bias_tables["category"]

In [ ]:
best_skus, worst_skus = best_and_worst_skus(enriched, n=20)
best_skus.to_csv(TABLE_DIR / "best_skus.csv", index=False)
worst_skus.to_csv(TABLE_DIR / "worst_skus.csv", index=False)
worst_skus.head()

In [ ]:
error_heatmap(store_category_metrics, x="category", y="store_id", output_path=str(OUTPUT_DIR / "store_category_wape_heatmap.png"))
actual_vs_predicted_scatter(enriched, output_path=str(OUTPUT_DIR / "actual_vs_predicted_scatter.png"))
worst_row = worst_skus.iloc[0]
worst_sku_time_series(
    enriched,
    store_id=str(worst_row["store_id"]),
    sku_id=str(worst_row["sku_id"]),
    output_path=str(OUTPUT_DIR / "worst_sku_time_series.png"),
)
sorted(path.name for path in OUTPUT_DIR.glob("*.png"))

## Where The Model Wins

High-volume smooth SKUs on non-promo days are expected to be the strongest segment. Fill this section with the lowest-WAPE volume and intermittency segments after running the notebook on the full LightGBM prediction artifact.

## Where The Model Loses

Intermittent/lumpy SKUs, promo days, and [category to fill] are expected weak spots. Replace the bracketed category with the highest-WAPE category from the generated segment tables.

## Systemic Biases

[categories from bias_flags] should be listed here with over/under-forecast direction and bias ratio.

## Recommendations For Iteration

Next steps: add richer promo timing/depth features, evaluate category-specific models, test quantile regression or Croston-style features for intermittent demand, and review high-bias store/category intersections with merchandising context.